## Setup

In [3]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()
API_URL = "https://pro.api.openmeasures.io"
jwt_token = os.getenv("OPEN_MEASURES_TOKEN")

_headers = {
    "Authorization": f"Bearer {jwt_token}"
}

In [13]:
# see quota
response = requests.get(f"{API_URL}/quota", headers=_headers)
data = response.json()

print(response.status_code)
print(f"{data['organization_usage']['core_api_monthly_requests_count'] / data['monthly_limits']['core_api_monthly_request_limit'] * 100:.2f}% of monthly quota used")
print(f"Info: {data}")

200
1.26% of monthly quota used
Info: {'organization_usage': {'current_year_month': '2026-06', 'last_active': '2026-06-15T19:14:09.573301Z', 'core_api_monthly_requests_count': 126, 'core_api_monthly_media_requests_count': 0, 'core_api_monthly_thumbnail_media_requests_count': 369, 'core_api_monthly_ai_moderation_requests_count': 0, 'core_api_monthly_entity_recognition_requests_count': 0, 'network_graph_monthly_requests_count': 0, 'network_graphs_count': 0, 'search_app_monthly_requests_count': 0, 'crawl_requests_usage': {'keyword': 0, 'profile': 0, 'telegram': 0, 'whatsapp': 0, 'channel': 0}, 'ai_credits_percentage': 0.0}, 'monthly_limits': {'core_api_global_access': False, 'core_api_global_media_access': False, 'core_api_monthly_request_limit': 10000, 'core_api_monthly_media_request_limit': 1000, 'core_api_monthly_thumbnail_media_request_limit': -1, 'core_api_monthly_ai_moderation_request_limit': 0, 'core_api_monthly_entity_recognition_request_limit': 0, 'core_api_time_cutoff_days': -1,

## Sample Statistics

In [5]:
# Import boolean query content
from query import ANTI, ISRAEL, PALESTINE

In [8]:
# Check and compare query sizes

a = ANTI
i = ISRAEL
p = PALESTINE

# dedupe is after sampling for fuzzy-identical cases
queries = {
    "ip_not": f'(({i}) OR ({p})) NOT ({a})',
    "ip_and": f'(({i}) OR ({p})) AND ({a})',
    "anti_only": f'({a}) NOT (({i}) OR ({p}))',
}

sites_to_test = ["4chan", "8kun", "truthsocial", "bluesky"]

results_summary = {}

for query_label, term_query in queries.items():
    results_summary[query_label] = {}
    for site in sites_to_test:
        test_params = {
            "sortdesc": "true",
            "limit": 1,
            "site": site,
            "term": term_query,
            "since": "2025-01-01",
            "until": "2025-12-31",
            "standard_fields": "true",
            "querytype": "boolean_content",
        }
        response = requests.get(f"{API_URL}/content", headers=_headers, params=test_params)
        print(f"====| {query_label} | {site} |====")
        print(f"Status: {response.status_code}")
        if response.status_code == 200:
            data = response.json()
            total_hits = data.get('total_hits')
            print(f"Total hits: {total_hits}")
            results_summary[query_label][site] = total_hits
        else:
            print(response.text[:300])
            results_summary[query_label][site] = None
        print()

print("\n=== SUMMARY ===")
for query_label, site_hits in results_summary.items():
    print(f"\n{query_label}:")
    for site, hits in site_hits.items():
        print(f"  {site}: {hits}")

====| ip_not | 4chan |====
Status: 200
Total hits: 726548

====| ip_not | 8kun |====
Status: 200
Total hits: 30521

====| ip_not | truthsocial |====
Status: 200
Total hits: 838506

====| ip_not | bluesky |====
Status: 200
Total hits: 6815668

====| ip_and | 4chan |====
Status: 200
Total hits: 68230

====| ip_and | 8kun |====
Status: 200
Total hits: 3094

====| ip_and | truthsocial |====
Status: 200
Total hits: 32643

====| ip_and | bluesky |====
Status: 200
Total hits: 49884

====| anti_only | 4chan |====
Status: 200
Total hits: 1080270

====| anti_only | 8kun |====
Status: 200
Total hits: 18587

====| anti_only | truthsocial |====
Status: 200
Total hits: 391832

====| anti_only | bluesky |====
Status: 200
Total hits: 404623


=== SUMMARY ===

ip_not:
  4chan: 726548
  8kun: 30521
  truthsocial: 838506
  bluesky: 6815668

ip_and:
  4chan: 68230
  8kun: 3094
  truthsocial: 32643
  bluesky: 49884

anti_only:
  4chan: 1080270
  8kun: 18587
  truthsocial: 391832
  bluesky: 404623


In [9]:
# Descriptive statistics

import pandas as pd

strata = ["ip_not", "ip_and", "anti_only"]

# results_summary structure: {query_label: {site: total_hits}}
df = pd.DataFrame(results_summary).T  # rows = query variant, columns = site
df.index.name = "query_variant"

# reconstruct 'full' as the sum of the three mutually exclusive partitions (sanity check + denominator)
df.loc["full"] = df.loc[strata].sum()

print("=== Raw hit counts per stratum ===")
print(df.to_string())

print("\n=== Composition: % of full corpus each stratum represents ===")
composition_df = (df.div(df.loc["full"], axis=1) * 100).round(2)
print(composition_df.to_string())

print("\n=== Composition by site ===")
for site in df.columns:
    print(f"\n{site}: total = {df.loc['full', site]:,}")
    for stratum in strata:
        pct = composition_df.loc[stratum, site]
        count = df.loc[stratum, site]
        print(f"  {stratum:<12} {count:>10,} ({pct:>5.2f}%)")

print("\n=== Cross-platform share: where does each stratum's volume come from? ===")
row_share_df = (df.div(df.sum(axis=1), axis=0) * 100).round(2)
print(row_share_df.loc[strata].to_string())

=== Raw hit counts per stratum ===
                 4chan   8kun  truthsocial  bluesky
query_variant                                      
ip_not          726548  30521       838506  6815668
ip_and           68230   3094        32643    49884
anti_only      1080270  18587       391832   404623
full           1875048  52202      1262981  7270175

=== Composition: % of full corpus each stratum represents ===
                4chan    8kun  truthsocial  bluesky
query_variant                                      
ip_not          38.75   58.47        66.39    93.75
ip_and           3.64    5.93         2.58     0.69
anti_only       57.61   35.61        31.02     5.57
full           100.00  100.00       100.00   100.00

=== Composition by site ===

4chan: total = 1,875,048
  ip_not          726,548 (38.75%)
  ip_and           68,230 ( 3.64%)
  anti_only     1,080,270 (57.61%)

8kun: total = 52,202
  ip_not           30,521 (58.47%)
  ip_and            3,094 ( 5.93%)
  anti_only        18,587 

## Stratified Sampling

In [ ]:
import pandas as pd

df = pd.json_normalize(data["results"])
df.to_parquet("test_query_results.parquet", engine="pyarrow", index=False)
print(f"Saved {len(df)} rows to test_query_results.parquet")

## Dedupe 
Exact by ID and Fuzzy with Hashing  
Independent of platform so the models (platform agnostic) don't overfit

In [ ]:
import pandas as pd

# combine all five strata pulls into one df first
all_dfs = []
for stratum_name, stratum_df in stratum_results.items():  # however storing each pull
    stratum_df["stratum"] = stratum_name
    all_dfs.append(stratum_df)

combined_df = pd.concat(all_dfs, ignore_index=True)
print(f"Combined before dedup: {len(combined_df)} rows")

combined_df = dedupe_by_id(combined_df, id_col="id")
combined_df = dedupe_by_exact_text_hash(combined_df, text_col="text")

print(f"Final deduped corpus: {len(combined_df)} rows")